# Running Ollama locally

[Ollama](https://ollama.com/) provides a convenient way to run large language models locally, abstracting away most of the complexity involved in model setup, hardware configuration, and inference management. It supports a wide range of models in the GGUF format and offers a consistent interface for model management, versioning, and execution. By exposing a local HTTP API compatible with common LLM workflows, Ollama integrates easily with external libraries and frameworks for tasks such as prompt engineering, retrieval‑augmented generation (RAG), and evaluation. This makes it especially suitable for usages where data privacy, reproducibility, or offline operation are important considerations.

## Local Installation of Ollama

Ollama can be installed locally with minimal setup. On **Linux** and **macOS**, it can be installed by running the following command in a terminal:

```bash
curl -fsSL https://ollama.com/install.sh | sh
```
On **Windows**, Ollama can be installed by downloading and running the official installer available on the https://ollama.com/.

After installation, Ollama is started with the command:
```bash
ollama serve
```
This launches a local API server listening on http://localhost:11434. Language models can then be downloaded using:
```bash
ollama pull
```
Models can be used either via the command line with:
```bash
ollama run <model-name>
```
or programmatically through the local HTTP API. Ollama automatically detects available GPU hardware and uses it when possible, while also supporting CPU-only systems without additional configuration.

Local Large Language Models and Retrieval‑Augmented Generation
---

The first part of this notebook will pull the Llama 3.2 model and set it up to enable text generation. We will then use it to generate answers using RAG methodology.

In [25]:
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 841 KB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 2.8 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 3.5 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   0% ▕                  ▏ 5.2 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏ 6.9 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏ 7.7 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕                  ▏ 9.3 MB/1.3 GB                  pulling manifest 
pulling 74701a8c35f6:   1% ▕         

In [2]:
!pip install dspy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Now we add our language model running in ollama to the dspy environment, so that the dspy library can use it to generate answers.

In [26]:
import dspy
lm = dspy.LM('ollama_chat/llama3.2:1b', api_base='http://localhost:11434', api_key='')
dspy.configure(lm=lm)

Now we use the GaMS model to answer a simple question.

In [27]:
print(lm("What is one purpose of sleep?"))

['One primary purpose of sleep is to allow the body and mind to repair and regenerate tissues, build bone and muscle, and strengthen the immune system. During sleep, the body produces proteins that help to repair damaged cells and build new ones, which can help to improve overall health and reduce the risk of chronic diseases.']


## Adding Context to a Query

In Retrieval-Augmented Generation (RAG), we improve the performance of language models by providing them with external context retrieved from a knowledge base. This is particularly useful when the model, by itself, doesn't have enough information to answer a question accurately.

The **dspy** system helps us chain together retrieval and generation in a modular way. Let's explore how to enhance query answering by supplying relevant context.

### Without Context

Let’s take a query that the model is unlikely to answer correctly without additional context.

**Query:**
>Who is the professor for the course Web Information Extraction and Retrieval?

This question is very specific. A general-purpose language model might not have this detail in its training data making it unlikely that its answer will be correct.

**Model's (possible) response (without context):**

> I\'m not aware of any specific information about a course called "Web Information Extraction and Retrieval" that I can provide an accurate answer to. There are many courses with similar names or topics, and without more context or details, it\'s difficult for me to pinpoint the exact professor or institution.\n\nIf you could provide more information about the course, such as:\n\n* The university or institution offering the course\n* The department or faculty that teaches the course\n* Any specific requirements or prerequisites for the course\n\nI\'ll do my best to help you find the information you\'re looking for.'

In [30]:
response = lm("Who is the professor for the course Web Information Extraction and Retrieval?")
print(response[0])

I'm not aware of any specific information about a course called "Web Information Extraction and Retrieval" that I can provide an accurate answer to. There are many courses with similar names or topics, and without more context or details, it's difficult for me to pinpoint the exact professor or institution.

If you could provide more information about the course, such as:

* The university or institution offering the course
* The department or faculty that teaches the course
* Any specific requirements or prerequisites for the course

I'll do my best to help you find the information you're looking for.


When asking the Llama model without context, it does not give us specific answer. We can improve this by providing a document containing the answer as the context to the model.

### With Retrieved Context

In this example, we will answer the question again, this time providing relevant context extracted from the [class website](https://www.fri.uni-lj.si/en/course/63551).

Lets first retrieve the website and clean it to gather the context. The implementation below is very crude, but should provide the required information.

In [1]:
import requests
from bs4 import BeautifulSoup

# URL of the page
url = "https://www.fri.uni-lj.si/en/course/63551"

# Send HTTP request
response = requests.get(url)
response.raise_for_status()  # Raise error if request failed
#print(response.text)

# Parse HTML content
soup = BeautifulSoup(response.text, 'html.parser')

# Extract the main content area (adjust based on actual HTML structure)
main_content = soup.find('div', class_='section-data')
#print(main_content)

# Extract plain text
plain_text = main_content.get_text(separator='\n', strip=True)
print(plain_text)


Contents
Web is almost an unlimited source of information. Using search engines such as
Google
,
Bing
and similar we can easily find web pages with possibly relevant information. The number of returned pages would usually however be very large which does not allow for manual processing. The solution to this are computer programs that are able to find and extract relevant information from possibly very large number of non-structured or semi-structured documents and return results in structured form.
COURSE GOAL
The main objective of this course is to teach students about how to develop programs for web search (including surface web and deep web search) and for extraction of structural data from both, static and dynamic web pages. Beside basic concepts of the web search and retrieval, students will learn about relevant techniques and approaches. After the course, if successful, students will be able to develop programs for automatic web search and structured data extraction from web page

**Adding the context to the query**:
The most basic way to add the context to the query is to write a simple string prompt that joins the question and the context together. We will use a prompt like this:
```
Answer the following question using context.
Context: {context}
Question: {question}
```

In [31]:
prompt = """
Answer the following question using context.
Context: {context}
Question: {question}
"""
print(lm(prompt.format(context=plain_text, question="Who is the professor for the course Web Information Extraction and Retrieval?")))

['The professor for the course "Web Information Extraction and Retrieval" is Prof. dr. Marko Bajec.']


**Expected output:**

> The professor for the course "Web Information Extraction and Retrieval" is Prof. dr. Marko Bajec.

This time, the model provides us with the correct answer. This demonstrates how **context retrieval** enables the model to answer questions it otherwise couldn't, which is the essence of the RAG approach.

### Using DsPY to generate a prompt
In the previous example, we manually wrote the prompt to include the context and the question for the model. While this approach worked in our case, it might require tweaking the prompt based on the model, context etc.

The DsPY library enables us to abstract the creation of the prompt templates by describing the task as a dspy.Signature class and defining the input and output data. The library then uses this information to generate prompt templates for queriing the large language model.

The DsPY library even allows us to "train" the prompts by trying different styles and evaluating results based on a scoring function. However, in this example, we will only use a simple predictor without any training.

### Class Definition

```python
class SimpleRAG(dspy.Signature):
    """Answer the user's question based on the context."""
    context: str = dspy.InputField(desc="Text context")
    question: str = dspy.InputField(desc="Question")
    answer: str = dspy.OutputField()
```

#### Components Breakdown

- **`class SimpleRAG(dspy.Signature)`**  
  This creates a new signature class for a DSPy module. A signature defines the expected input and output fields for a model program. This structure is used when chaining together modules like retrievers and generators. The comment is used to define the task for the model.

- **`context: str = dspy.InputField(desc="Text context")`**  
  An **input field** named `context`, typed as a string. The `desc` parameter is a human-readable description that can be used in documentation or debugging to clarify what the input represents. Here, it contains background information or retrieved documents that help the model answer the question.

- **`question: str = dspy.InputField(desc="Question")`**  
  Another **input field**, also a string, representing the user’s query. The description clarifies that this is the question to be answered based on the context.

- **`answer: str = dspy.OutputField()`**  
  This is the **output field** the model is expected to generate. It will contain the answer inferred from the given context and question.

In [32]:
# Create a simple RAG-style program
class SimpleRAG(dspy.Signature):
    """Answer the user's question based on the context."""
    context:str = dspy.InputField(desc="Text context")
    question:str = dspy.InputField(desc="Question")
    answer:str = dspy.OutputField()

rag_module = dspy.Predict(SimpleRAG)

# Example inputs
context = plain_text
question = "Who is the professor for the course Web Information Extraction and Retrieval?"

# Get prediction
response = rag_module(context=context, question=question)
print(response.answer)

Prof. dr. Marko Bajec


**Expected output:**

> Prof. dr. Marko Bajec

Again, we get the correct response from the model, showcasing how the DsPY library can be used to answer a question using context.

### What about Slovenian language?

Although most open‑weight language models are predominantly trained on high‑resource languages such as English, Slovenian can still be handled effectively, especially when combining local inference with external context. Several community‑released models already provide reasonable support for Slovenian, for example 

- [GaMS‑9B‑Instruct](https://huggingface.co/cjvt/GaMS-9B-Instruct) (for Ollama, use this GGUF version [hf.co/tknez/GaMS-9B-Instruct-GGUF:Q4_K_M](https://huggingface.co/tknez/GaMS-9B-Instruct-GGUF)) and 
- [Salamandra‑2B‑Instruct](https://huggingface.co/BSC-LT/salamandra-2b-instruct) (for Ollama, use this GGUF version [hf.co/robbiemu/salamandra-2b-instruct:Q4_K_M](https://huggingface.co/robbiemu/salamandra-2b-instruct)), 

both of which can be run locally via Ollama. While their zero‑shot generative quality may vary, particularly for morphologically complex constructions, Retrieval‑Augmented Generation significantly improves factual accuracy by grounding the answers in Slovenian source documents. This makes local RAG pipelines a practical and privacy‑preserving approach for experimenting with Slovenian language tasks.